# The Three Pillars - Instruction, Context, and Constraints

Welcome! This notebook brings our most important prompt design framework to life: **The Three Pillars**. As we discussed, nearly every effective prompt is a combination of these three components:

1. **Instruction:** What you want the model to *do*. (e.g., "Write unit tests.")
2. **Context:** The information the model *needs* to perform the instruction. (e.g., The code to be tested.)
3. **Constraints:** The *rules* the output must follow. (e.g., "Use the `unittest` framework" or "Format as JSON.")

To see this in action, we'll perform a common developer task: **generating unit tests for a Python function.** We will build our prompt iteratively, starting with just an instruction and adding pillars one by one to see how the quality of the output dramatically improves.

## The Function to be Tested (Our Context)

First, let's define the code we want to test. This Python function will serve as the **Context** for our prompt later on. It's designed to find the longest string in a list and includes a few edge cases we'll want to test.

In [1]:
# This is the function we will ask the LLM to write tests for.

def find_longest_string(strings: list[str]) -> str | None:
    """
    Given a list of strings, finds and returns the longest string.

    - If the list is empty, it returns None.
    - If there's a tie, it returns the first one found.
    - It raises a TypeError if the list contains non-string elements.
    """
    if not strings:
        return None

    longest_string = ""
    for s in strings:
        if not isinstance(s, str):
            raise TypeError("All elements in the list must be strings.")
        if len(s) > len(longest_string):
            longest_string = s

    return longest_string

In [2]:
import inspect
function_source_code = inspect.getsource(find_longest_string)

print("--- Function to be tested ---")
print(function_source_code)

--- Function to be tested ---
def find_longest_string(strings: list[str]) -> str | None:
    """
    Given a list of strings, finds and returns the longest string.

    - If the list is empty, it returns None.
    - If there's a tie, it returns the first one found.
    - It raises a TypeError if the list contains non-string elements.
    """
    if not strings:
        return None

    longest_string = ""
    for s in strings:
        if not isinstance(s, str):
            raise TypeError("All elements in the list must be strings.")
        if len(s) > len(longest_string):
            longest_string = s

    return longest_string



## Setup: Reusable Helper Functions

In [14]:
import os
import litellm
#from dotenv import load_dotenv
from textwrap import dedent

#load_dotenv()

MODEL_NAME = "ollama/qwen2.5:3b"
MAX_TOKENS_DEFAULT = 400

def get_completion(
    prompt,
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS_DEFAULT,
    **kwargs
):
    parsed_messages = []

    if type(prompt) is str:
        parsed_messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        parsed_messages = prompt

    response = litellm.completion(
        model=model,
        messages=parsed_messages,
        max_tokens=max_tokens,
        **kwargs
    )

    return response.choices[0].message.content

print("Helper functions are defined and ready.")

Helper functions are defined and ready.


## Attempt 1: Using Only an Instruction (Pillar 1)

Let's start with the most basic prompt possible: just an instruction. We'll ask the model to perform a task without giving it any of the necessary information.

As you'll see, the model can't read our minds. Without context, the instruction is ambiguous and the model has to guess, often resulting in a generic, unhelpful, or incomplete response.

In [15]:
prompt_v1 = "Write unit tests."

response_v1 = get_completion(prompt_v1)

In [16]:
print(response_v1)

Writing unit tests depends on the programming language and the system you're testing. I'll provide an example in Python, as it's widely used and has a robust testing framework, `unittest` or `pytest`.

Let's consider a simple example where we have a function to perform some arithmetic operation. We'll write tests to ensure that the function behaves correctly.

Here is an example function and its corresponding unit tests:

```python
# Define the function
def add(a, b):
    return a + b

# Now, let's write unit tests
import unittest

class TestAddFunction(unittest.TestCase):
    def test_add_positive_numbers(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -2), -3)

    def test_add_mixed_numbers(self):
        self.assertEqual(add(-1, 1), 0)

    def test_add_zero(self):
        self.assertEqual(add(0, 5), 5)

    def test_add_floats(self):
        self.assertAlmostEqual(add(1.5, 2.5), 4.0)

    def test_add_large_n

## Attempt 2: Adding Context (Pillars 1 & 2)

Now, let's provide the second pillar: **Context**. We'll give the model the same instruction, but this time we'll also provide the source code of the function we want it to test.

This is a huge improvement. The model now has the necessary information to perform the task. However, as we'll see, without constraints, the output might not be exactly what we need for our project.

In [17]:
prompt_v2 = dedent(f"""
Write unit tests for the following Python function:

```python
{function_source_code}
```
""")

response_v2 = get_completion(prompt_v2)

In [18]:
print(response_v2)

To write unit tests for the `find_longest_string` function, we need to cover a variety of scenarios. These scenarios include edge cases like an empty list, a list with no duplicate longest string, and a list with duplicate longest string. We should also cover cases where the list is not provided as expected (i.e., a non-list type) and handle the TypeError correctly.

Here are the unit tests using the `unittest` framework for Python:

```python
import unittest

def find_longest_string(strings: list[str]) -> str | None:
    """
    Given a list of strings, finds and returns the longest string.

    - If the list is empty, it returns None.
    - If there's a tie, it returns the first one found.
    - It raises a TypeError if the list contains non-string elements.
    """
    if not strings:
        return None

    longest_string = ""
    for s in strings:
        if not isinstance(s, str):
            raise TypeError("All elements in the list must be strings.")
        if len(s) > len(lo

## Attempt 3: Adding Constraints for Precision (Pillars 1, 2, & 3)

This is the final and most powerful step. We will add the third pillar: **Constraints**. We will provide the same instruction and context, but now we'll add a specific set of rules that the output must follow. This gives us precise control over the final result.

In [24]:
prompt_v3 = dedent(f"""
## Instruction
Write unit tests for the following Python function. Write a short 
comment above each test to clarify the test case that is currently being tested.

## Context
```python
{function_source_code}
```

## Requirements
* Use pytest, not unittest
* Do not repeat words within the same test
* Create a test class named `TestFindLongestString`
* Include at least 4 test cases:
  * A standard case with a clear longest string.
  * A case where the input list is empty.
  * A case to test tie-breaking behaviour (different words with the same length).
  * A case for the TypeError
* If there are any additional edge cases, pelase list them and include a test for each
* The output must ONLY be a single, complete block of Python code
* Do not include any explanations or surrounding text
""")

response_v3 = get_completion(prompt_v3)

In [25]:
print (response_v3)

```python
import pytest

class TestFindLongestString:
    def test_standard_case(self):
        """Test with a list containing a few strings of different lengths."""
        strings = ["hello", "world", "alibaba"]
        assert find_longest_string(strings) == "alibaba"

    def test_empty_list(self):
        """Test the case when the input list is empty."""
        strings = []
        assert find_longest_string(strings) is None

    def test_tie_breaker(self):
        """Test the case where all strings have the same length."""
        strings = ["abc", "def", "ghi"]
        assert find_longest_string(strings) == "abc"

    def test_type_error(self):
        """Test the case when the input list contains non-string elements."""
        strings = ["abc", 123, "def"]
        with pytest.raises(TypeError):
            find_longest_string(strings)

    def test_single_string(self):
        """Test with a single string in the list."""
        strings = ["test"]
        assert find_longest_s